In [0]:
%run ./utils

# Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load

In [0]:
dbutils.widgets.text("silver_table_name", "tb_atendentes")
silver_table_name = dbutils.widgets.get("silver_table_name")

df_atendentes = load_df(silver_table_name)


# Atributos


In [0]:
# surrogate key (big int)
df_dim_atendentes = add_sequential_key(df_atendentes, "atendente_sk")

In [0]:
df_dim_atendentes = (
    df_dim_atendentes
        .withColumn("nivel_atendimento", F.col("nivel_atendimento").cast("int"))
        .withColumn("data_admissao", F.to_date("data_admissao"))
        .withColumn("data_desligamento", F.to_date("data_desligamento"))
        .select(
            "atendente_sk",
            "id_atendente",
            "nome_atendente",
            "nivel_atendimento",
            "data_admissao",
            "data_desligamento",
            "status_atendente"
        )
)

display(df_dim_atendentes)

# Verificação

In [0]:
# dimensão deve ter uma linha por atendente (SCD tipo 1: valor anterior substituído pelo novo) e a chave deve ser única
total = df_dim_atendentes.count()
atendentes = df_dim_atendentes.select("id_atendente").distinct().count()
chaves = df_dim_atendentes.select("atendente_sk").distinct().count()
print(f"Linhas: {total} | id_atendente únicos: {atendentes} | atendente_sk únicas: {chaves}")

# dim deve ter a mesma quantidade de atendentes da silver (25)
total_silver = df_atendentes.count()
print(f"Linhas na Silver: {total_silver} | Linhas na dimensão: {total}")

# nível deve ser 1 ou 2 (o cast para int anula valores inválidos 
display(df_dim_atendentes.groupBy("nivel_atendimento").count())

In [0]:
dbutils.widgets.text("gold_table_name", "dim_atendente")
gold_table_name = dbutils.widgets.get("gold_table_name")

save_df(df_dim_atendentes, gold_table_name)